# Day 05 课堂作业

**课程：** 电商用户行为数据分析 — 指标口径、用户画像与流失分析
**日期：** 2026-07-15

---


## 问题1：写出总体流失率公式并说明分母


**流失率公式：**

总体流失率 = 流失用户数 / 总用户数

因为 Churn 列只取 0 和 1，也可写作：

总体流失率 = Churn.mean()

**分母说明：**

分母是全体 **5,630 名用户** 的总数（即 `df["CustomerID"].nunique()`）。

当前数据中：
- 分子（流失人数）= **948 人**
- 分母（总用户数）= **5,630 人**
- 总体流失率 = 948 / 5,630 = **16.84%**

> 关键点：分组流失率的分母是该分组的用户数，而不是全体用户数。例如「0-1年用户流失率」的分母是 3,552（0-1年用户总数），而不是 5,630。


## 问题2：从生命周期或投诉分析中选择一条数据发现


### 选择：生命周期（TenureGroup）分析

| 生命周期 | 用户数 | 流失人数 | 流失率 |
|----------|--------|----------|--------|
| 0-1年   | 3,552  | 846      | **23.82%** |
| 1-3年   | 2,074  | 102      | 4.92%   |
| 3-5年   | 3      | 0        | 0.00%   |

**数据发现（规范结论模板）：**

在 **0-1年新用户** 中，**流失率** 为 **23.82%**，与 **1-3年用户（4.92%）** 相比 **高出约 4.8 倍**。当前样本显示 **用户生命周期长度与流失率** 存在明显负向关联，可能与 **新用户尚未建立使用习惯、平台黏性不足** 有关；仍需结合 **新用户首单体验、品类偏好分布和优惠券使用行为** 进一步验证。

> 注：3-5年组仅有 3 人，样本量过小，该组流失率不具统计参考意义。


## 问题3：把一条因果化表述改写为「关联 + 待验证」的表达


**因果化表述（不当）：**

> 「投诉必然导致用户流失。」

**改写为「关联 + 待验证」的表达（合理）：**

> 当前样本中，**有投诉用户的流失率为 31.67%，无投诉用户的流失率为 10.93%**，投诉与流失之间存在明显关联。但这尚不能说明投诉是流失的原因——也可能是流失倾向较高的用户本身更容易投诉（反向因果），或两者同时受第三方因素（如订单体验差）影响。仍需排除反向因果及其他混淆因素后，结合用户投诉内容、投诉处理结果和时间先后关系进一步验证。

**对照表：**

| 投诉状态 | 用户数 | 流失人数 | 流失率  | 平均满意度 |
|----------|--------|----------|---------|-----------|
| 无投诉   | 4,026  | 440      | 10.93%  | 3.09      |
| 有投诉   | 1,604  | 508      | 31.67%  | 3.00      |

> 值得注意：投诉用户的平均满意度（3.00）与无投诉用户（3.09）差距不大，说明满意度评分与投诉行为的关系并不简单，需要更细致的分析。


## 问题4：说明当前数据为什么不能计算GMV或月度趋势


当前数据 **缺少两个关键字段**：

### 1. 缺少订单金额 → 无法计算 GMV

GMV（Gross Merchandise Volume，总商品交易额）需要每笔订单的实付金额。当前数据中：
- 只有 `CashbackAmount`（**返现金额**），是平台返还给用户的，不等于消费金额；
- 只有 `OrderCount`（订单数量），没有每笔订单的金额；
- 因此最多只能讨论「返现行为差异」，不能计算销售额、GMV 或客单价。

### 2. 缺少订单日期 → 无法计算月度趋势

月度趋势分析（如月活、月度 GMV 走势）需要每笔订单的时间戳。当前数据中：
- 只有 `DaySinceLastOrder`（距上次下单天数），是一个**快照值**，无法还原历史时间线；
- 没有订单日期字段，无法按年/月分组聚合；
- 因此只能做横截面分析（当前快照），不能做时间趋势分析。

### 总结

| 想计算     | 需要的字段       | 当前有的字段                       | 能否算 |
|------------|-----------------|-----------------------------------|--------|
| GMV        | 订单金额         | CashbackAmount（返现，不等于金额）  | 不能   |
| 客单价     | 订单金额、订单数 | 仅有 OrderCount                   | 不能   |
| 月度趋势   | 订单日期         | DaySinceLastOrder（快照值）        | 不能   |
| 流失率     | Churn、CustomerID | Churn、CustomerID               | 能     |
| 平均订单数 | OrderCount       | OrderCount                        | 能     |


---

## 附录：关键数据验证

以下为本次作业所用数据的基础验证结果（来源：Day 04 清洗输出）：

- 数据形状：5,630 行 x 22 列
- CustomerID 重复数：0（每行代表一名独立用户）
- 核心字段缺失数：0
- Churn 取值：[0, 1]
- 总体流失率：**16.84%**
